## Worldwide multi-location HOPP example
---
This example shows how to simulate a hybrid renewable energy plant using HOPP for any one (onshore) location or set of locations in the world.

We download solar resource data from NSRDB (through NREL API) and wind resource data from the Open-Meteo database. 

Unlike the NREL Wind Toolkit database, the Open-Meteo database provides worldwide coverage, though the resulting data require some modification to be usable in HOPP.

### Import required modules
We start by importing the necessary HOPP modules and other packages that will be needed later.

In [ ]:
# Import HOPP modules
from hopp.simulation.hopp_interface import HoppInterface
from hopp.tools.dispatch.plot_tools import plot_battery_output, plot_generation_profile
# Import other packages
import pandas as pd
import numpy as np
import os
import datetime
import time

### Set NREL API key
To access the NSRDB (solar resource) data, we need to set an API key. You can obtain an API key from the [NREL developer website](https://developer.nrel.gov/signup/).

**IMPORTANT!** Enter below your own API key and associated email address

In [ ]:
api_key = 'your-api-key'
email_address = 'your-email-address'

### Define the supporting functions

(1) A function that uses the NREL API to request NSRDB (solar resource) data for a set of user-specified coordinates and saves it as a CSV file in a user-specified directory.

In [3]:
import requests

# Function to get solar data via NREL API for one location
# lat = latitude, lon = longitude, name = location name (e.g. city, village)
def nrel_query(lat, lon, name, solar_output_dir):
    
    # URL for the NREL Meteosat Prime Meridian TMY dataset API
    NSRDB_URL = f"https://developer.nrel.gov/api/nsrdb/v2/solar/nsrdb-msg-v1-0-0-tmy-download.csv?api_key={api_key}&wkt=POINT({lat} {lon})&attributes=ghi,dhi,dni,wind_speed,air_temperature,solar_zenith_angle,surface_pressure,dew_point&names=tmy-2022&utc=false&leap_day=false&interval=60&email={email_address}"
    
    # Send request to the NREL API and save response
    print(f"\nSending request to NREL API for location with lat {lat} and lon {lon}...")
    response = requests.get(NSRDB_URL)

    # Raise an HTTPError for bad responses (4XX or 5XX client/server errors)
    response.raise_for_status()
    print(f"Received HTTP {response.status_code} response from API.")

    response_text = response.text

    # Check for API-specific errors within the CSV content even if status is 200
    first_few_lines = response_text.splitlines()
    is_api_error = False
    if first_few_lines:
        # Check if the first line (potential header of an error CSV) contains "error message"
        if "error message" in first_few_lines[0].lower():
                is_api_error = True

    if is_api_error:
        print(f"API Error: Received an error message from NREL API within the data response:")
        print("-------------------- API ERROR RESPONSE (first 5 lines) --------------------")
        print("\n".join(first_few_lines[:5])) # Print first 5 lines of the error
        print("--------------------------------------------------------------------------")

    # Set up output directory
    os.makedirs(solar_output_dir, exist_ok=True)

    # Define output filename
    filename = f"{lat}_{lon}_{name}_NSRDB.csv"
    filepath = os.path.join(solar_output_dir, filename)

    # Save the data
    print(f"INFO: Saving data to {filepath}...")
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        f.write(response_text)

    print(f"SUCCESS! Downloaded and saved solar data to: {filepath}")

    # Delay for 1 second to prevent API rate limiting
    time.sleep(1)

    return filepath

(2) A function that queries the Open-Meteo database for wind data for a set of user-specified coordinates and saves it as a CSV file in a user-specified directory.

In [4]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Function to get wind data via OpenMeteo API for one location
# lat = latitude, lon = longitude, name = location name (e.g. city, village)
def openmeteo_query(lat, lon, name, wind_output_dir):
    
    # Set up the OpenMeteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)

    # Make sure all required weather variables are listed here (modify as needed)
    # The order of variables in hourly or daily is important to assign them correctly below
    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, 
        "longitude": lon,
        "start_date": "2022-01-01",
        "end_date": "2022-12-31",
        "hourly": ["temperature_80m", "wind_speed_80m", "wind_direction_80m", "temperature_120m", "wind_speed_120m", "wind_direction_120m"],
        "timezone": "auto",
        "wind_speed_unit": "ms"
    }
    # Open-Meteo API query
    print(f"\nSending request to Open-Meteo API for location with lat {lat} and lon {lon}...")
    responses = openmeteo.weather_api(url, params=params)

    # Output directory setup
    os.makedirs(wind_output_dir, exist_ok=True)

    # Print data from API query
    response = responses[0]
    print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation {response.Elevation()} m asl")
    print(f"Timezone {response.Timezone()}{response.TimezoneAbbreviation()}")
    print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_80m = hourly.Variables(0).ValuesAsNumpy()
    hourly_wind_speed_80m = hourly.Variables(1).ValuesAsNumpy()
    hourly_wind_direction_80m = hourly.Variables(2).ValuesAsNumpy()
    hourly_wind_speed_120m = hourly.Variables(3).ValuesAsNumpy()
    hourly_wind_direction_120m = hourly.Variables(4).ValuesAsNumpy()
    hourly_temperature_120m = hourly.Variables(5).ValuesAsNumpy()

    hourly_data = {"date": pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
        end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
        freq = pd.Timedelta(seconds = hourly.Interval()),
        inclusive = "left"
    )}

    hourly_data["temperature_80m"] = hourly_temperature_80m
    hourly_data["wind_speed_80m"] = hourly_wind_speed_80m
    hourly_data["wind_direction_80m"] = hourly_wind_direction_80m
    hourly_data["wind_speed_120m"] = hourly_wind_speed_120m
    hourly_data["wind_direction_120m"] = hourly_wind_direction_120m
    hourly_data["temperature_120m"] = hourly_temperature_120m

    hourly_dataframe = pd.DataFrame(data = hourly_data)
    print(hourly_dataframe.head())
    printable_data = hourly_dataframe.to_string()

    # Define output filename
    filename = f"{response.Latitude()}_{response.Longitude()}_{response.Elevation()}_{name}_openmeteo_wind.csv"
    filepath = os.path.join(wind_output_dir, filename)

    # Save the data
    print(f"INFO: Saving data to {filepath}...")
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        f.write(printable_data)

    print(f"SUCCESS! Downloaded and saved wind data to: {filepath}")

    # Delay for 1 second to prevent API rate limiting
    time.sleep(1)

    return filepath

(3) A function that converts the wind data obtained through Open-Meteo to a HOPP-friendly format.

In [5]:
# Function converts a OpenMeteo-formatted CSV file to a WTK-formatted SRW file.
def convert_to_srw(input_filepath, output_dir):
    
    try:
        # --- 1. Data Import and Initial Manipulation ---
        print("\nConverting OpenMeteo-formatted CSV file to WTK-formatted SRW file...")
        print(f"Processing file: {input_filepath}")
        df = pd.read_csv(input_filepath, sep='\s+')

        # --- 2. Column Modification ---
        # Reordering the columns (if needed).
        df = df[[
            'date', 'temperature_80m', 'wind_speed_80m','wind_direction_80m', 
            'wind_speed_120m', 'wind_direction_120m', 'temperature_120m'
        ]]
        # Delete the 'date' column.
        df = df.drop(columns=['date'])
        # Insert a column of zeros at position 2
        df.insert(1, 'pressure_80m', 1.0)
        # Insert another column of zeros at position 6
        df.insert(5, 'pressure_120m', 1.0)
        # Rounding to 3 decimals
        df = df.round(4)
        # Convert to numpy array
        data = df.to_numpy()

        # --- 3. File Output ---
        # Create the full path for the new .srw file.
        base_filename = os.path.basename(input_filepath)
        filename_without_ext = os.path.splitext(base_filename)[0]
        split_filename = filename_without_ext.split("_")
        latitude = round(float(split_filename[0]), 4)
        longitude = round(float(split_filename[1]), 4)
        altitude = round(float(split_filename[2]), 1)
        name = str(split_filename[3])
        output_filepath = os.path.join(output_dir, f"{latitude}_{longitude}_{altitude}_{name}_openmeteo_wind_converted.srw")

        # Define the 5-line header similar to the wtk_file format.
        header_lines = [
            f"123456,{name},state??,Country,2022,{latitude},{longitude},Not Available,1,8760\n",
            "WIND Toolkit data converted from OpenMeteo data\n",
            "Temperature,Pressure,Speed,Direction,Temperature,Pressure,Speed,Direction\n",
            "C,atm,m/s,Degrees,C,atm,m/s,Degrees\n",
            "80,80,80,80,120,120,120,120\n"
        ]

        # Write the header and the modified data to the new file.
        with open(output_filepath, 'w') as fi:
            # Write the custom 5-line header
            fi.writelines(header_lines)
            # Append the DataFrame to the file as comma-delimited, without its own header.
            #df.to_csv(fi, index=False, header=False, sep=',')
            np.savetxt(fi, data, fmt='%.4f', delimiter=',')

        print(f"Successfully converted to: {output_filepath}\n")

    except FileNotFoundError:
        print(f"Error: Input file not found at {input_filepath}\n")

    except Exception as e:
        print(f"An error occurred while processing {input_filepath}: {e}\n")

    return output_filepath

(4) A function that modifies the HOPP configuration YAML file to enable sequential HOPP simulations for multiple locations.

In [6]:
import ruamel.yaml

def yaml_modifier(file_path, new_lat, new_lon, solar_file, wind_file):

    # YAML parser
    yaml = ruamel.yaml.YAML()
    yaml.preserve_quotes = True 

    # Load the YAML file
    with open(file_path, 'r') as file:
        data = yaml.load(file)

    # Locate and modify the data
    data['site']['solar_resource_file'] = solar_file
    data['site']['wind_resource_file'] = wind_file
    data['site']['data']['lat'] = new_lat
    data['site']['data']['lon'] = new_lon

    # Write the changes back to the file
    with open(file_path, 'w') as file:
        yaml.dump(data, file)

    print(f"Successfully updated '{file_path}' with data for ({new_lat},{new_lon})")

(5) A function that runs the HOPP simulation for all the locations provided in an excel file. 

To run for one location, set **RANGE = [location_index]** where location_index = 0, 1, 2, ...

The HOPP results are saved in a separate txt file for each location. A single txt file with key results from all locations is also saved at the end. 

In [ ]:

def hopp_simulation(coord_file, yaml_file, solar_dir, wind_dir, hopp_dir):
    '''
    coord_file: Excel file containing locations and coordinates
    yaml_file: Yaml file containing HOPP configuration
    solar_dir: Directory containing solar data files
    wind_dir: Directory containing wind data files
    hopp_dir: Directory for HOPP output
    '''
    # Get today's date
    time_now = datetime.date.today()

    # Import locations and coordinates
    df = pd.read_excel(coord_file)
    lat = df['latitude'].to_numpy()
    lon = df['longitude'].to_numpy()
    name = df['location'].to_numpy()

    # Identify locations to simulate
    # For first location: RANGE = [0], for third location: RANGE = [2], or for all locations: RANGE = range(len(name))
    RANGE = range(len(name))

    # Initialize arrays for saving key results
    lcoe_data = np.zeros(len(RANGE))
    energy_data = np.zeros(len(RANGE))
    cf_data = np.zeros(len(RANGE))
    k = 0 #index for lcoe array

    # Iterate for all locations specified in RANGE
    for idx in RANGE:
        latitude = np.round(lat[idx], 2)
        longitude = np.round(lon[idx], 2)
        LAT = latitude.item() #convert np.array to float
        LON = longitude.item()
        LOCATION = name[idx]

        # Get solar data from NREL API
        SOLAR_FILE = nrel_query(LAT, LON, LOCATION, solar_dir)

        # Get wind data from OpenMeteo API
        WIND_FILE = openmeteo_query(LAT, LON, LOCATION, wind_dir)

        # Convert wind data to SRW format
        CONVERTED_WIND_FILE = convert_to_srw(WIND_FILE, wind_dir)

        # Modify YAML file for desired location
        yaml_modifier(yaml_file, LAT, LON, SOLAR_FILE, CONVERTED_WIND_FILE)

        #HOPP simulation
        hi = HoppInterface(yaml_file)
        print("\nHOPP simulation in progress...")
        hi.simulate(project_life=20) #specify the project lifetime in years

        hybrid_plant = hi.system
        print("\nPrinting and saving HOPP output...")

        #HOPP Output
        if not os.path.exists(hopp_dir):
            os.makedirs(hopp_dir)
        output_file = os.path.join(hopp_dir, f"hopp_output_{LAT}_{LON}_{LOCATION}_{time_now}.txt")

        Wind_eff = hybrid_plant.wind.value("annual_energy") / hybrid_plant.wind.value("annual_gross_energy")
        energy = hybrid_plant.annual_energies
        cf = hybrid_plant.capacity_factors
        lcoe_n = hybrid_plant.lcoe_nom
        lcoe_r = hybrid_plant.lcoe_real

        # Print to file
        with open(output_file, "w") as log:
            print(f"Wind output efficiency:\n {Wind_eff} \nAnnual Energies (kWh):\n {energy} \nCapacity factors (%):\n {cf} \nNominal LCOE (c/kWh):\n {lcoe_n} \nReal LCOE (c/kWh):\n {lcoe_r}"
                , file=log)
        print(f"HOPP output saved in '{output_file}'\n")

        # Save key results for this location
        lcoe_data[k] = lcoe_r["hybrid"]
        energy_data[k] = energy["hybrid"]
        cf_data[k] = cf["hybrid"]
        k += 1
        
        # Print to terminal
        print("Wind output efficieny:", Wind_eff)
        print("\nAnnual Energies (kWh):")
        print(energy)
        print("\nCapacity factors (%):")
        print(cf)
        print("\nNominal LCOE (cents/kWh):")
        print(lcoe_n)
        print("\nInflation-adjusted LCOE (cents/kWh):")
        print(lcoe_r)

        # Plot generation and battery dispatch profiles (We advise to comment these out if running multiple locations)
        #plot_generation_profile(hybrid_plant)
        #plot_battery_output(hybrid_plant)

    # Save key results
    data_stack = np.column_stack([name[RANGE], energy_data, cf_data, lcoe_data])
    lcoe_file = os.path.join(hopp_dir, f"LCOE_output_{time_now}.txt")
    with open(lcoe_file, "w") as log:
            print(f"Location, Annual generation (kWh), Capacity factor (%), Real LCOE (cents/kWh)\n {data_stack}", file=log)
    print(f"\nKey HOPP results saved in '{lcoe_file}'")


### Run the main code
Specify the excel file containing the locations' coordinates and names, the HOPP configuration YAML file, the directories containig solar and wind resource data files, and the output directory. To assist the user, an excel file is provided with coordinates for locations in western Africa.

Then, run the code below to simulate a hybrid renewable energy plant in all desired locations. 

Check the results in the output directory.

In [8]:
# INPUT FILES (MODIFY FILE PATHS AS NEEDED)
# Excel file with at least three columns (latitude, longitude, and location)
COORD_FILE = './inputs/west_africa_coordinates.xlsx'
# YAML file for HOPP configuration
YAML_FILE = "./inputs/03-wind-solar-battery.yaml"

# Directories for weather data and HOPP output (modify as needed)
SOLAR_DIR = './inputs/solar_data/'
WIND_DIR = './inputs/wind_data/'
HOPP_DIR = './hopp_output/'

# Run HOPP simulation for all locations
hopp_simulation(COORD_FILE, YAML_FILE, SOLAR_DIR, WIND_DIR, HOPP_DIR)

print("\nSUCCESS! HOPP simulation completed for all locations and results saved.")


Sending request to NREL API for location with lat 34.92 and lon -6.18...
Received HTTP 200 response from API.
INFO: Saving data to ./inputs/solar_data/34.92_-6.18_Larache_NSRDB.csv...
SUCCESS! Downloaded and saved solar data to: ./inputs/solar_data/34.92_-6.18_Larache_NSRDB.csv

Sending request to Open-Meteo API for location with lat 34.92 and lon -6.18...
Coordinates 34.9375°N -6.1875°E
Elevation 85.0 m asl
Timezone b'Africa/Casablanca'b'GMT+1'
Timezone difference to GMT+0 3600 s
                       date  temperature_80m  wind_speed_80m  \
0 2021-12-31 23:00:00+00:00        18.973000        1.780449   
1 2022-01-01 00:00:00+00:00        18.622999        1.920937   
2 2022-01-01 01:00:00+00:00        18.172998        3.220248   
3 2022-01-01 02:00:00+00:00        17.823000        3.764306   
4 2022-01-01 03:00:00+00:00        17.372999        3.360060   

   wind_direction_80m  wind_speed_120m  wind_direction_120m  temperature_120m  
0          128.157272        19.172998          